In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("..\\Data\\04-03_이상의_정의_진동데이터.csv", encoding="utf-8")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   일자      120 non-null    int64  
 1   진동RMS   120 non-null    float64
 2   부하율     120 non-null    float64
 3   가동여부    120 non-null    int64  
dtypes: float64(2), int64(2)
memory usage: 3.9 KB


In [6]:
df.head(10)

,일자,진동RMS,부하율,가동여부
0,0,3.14,65.0,1
1,1,3.53,71.1,1
2,2,3.24,71.4,1
3,3,3.57,70.4,1
4,4,3.39,74.2,1
5,5,3.56,70.6,1
6,6,3.52,74.9,1
7,7,3.87,79.4,1
8,8,3.26,64.3,1
9,9,2.88,59.1,1


In [ ]:
df_op = df[df["가동여부"] == 1]
df_op.head(10)

In [13]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 그래프 1 - 시간에 따른 진동 변화
plt.figure(figsize=(12, 4))
plt.plot(df_op["일자"], df_op["진동RMS"], linewidth=1.5, label="진동RMS")
plt.axhline(4.5, color="red", linestyle="--", alpha=0.7, label="고정 임계 4.5")
plt.xlabel("일자")
plt.ylabel("진동RMS (mm/s)")
plt.title("가동 구간 진동RMS 추이")
plt.legend()
plt.tight_layout()

# 이미지 저장 - show() 보다 먼저 호출해야 한다
# show()가 도화지를 비우기 때문에 순서가 바뀌면 빈 파일이 저장된다
plt.savefig("04-03_graph1_vibration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 그래프 2 - 부하율과 진동RMS의 관계 (80일 미만 가동 데이터)
df_norm = df[(df["가동여부"] == 1) & (df["일자"] < 80)]
print("정상 후보:", len(df_norm), "행")

plt.figure(figsize=(6, 5))
plt.scatter(df_norm["부하율"], df_norm["진동RMS"], alpha=0.6)
plt.xlabel("부하율 (%)")
plt.ylabel("진동RMS (mm/s)")
plt.title("부하율 vs 진동RMS (일자 < 80)")
plt.tight_layout()
plt.savefig("04-03_graph2_load_vs_vib.png", dpi=150, bbox_inches="tight")
plt.show()

print("상관계수:", round(df_norm["부하율"].corr(df_norm["진동RMS"]), 3))

In [ ]:
# [4] 정상 기준 숫자로 확인하기
mean_v = df_norm["진동RMS"].mean()
std_v = df_norm["진동RMS"].std()
stat_th = mean_v + 3 * std_v

print("정상 후보 데이터 수 :", len(df_norm), "행")
print("진동RMS 평균        :", round(mean_v, 3), "mm/s")
print("진동RMS 표준편차    :", round(std_v, 3))
print("통계 임계치(평균+3σ):", round(stat_th, 3), "mm/s")

# 고정 임계치와 통계 임계치의 검출 결과 비교
for name, th in [("고정 4.5", 4.5), (f"통계 {stat_th:.2f}", stat_th)]:
    over = df_op[df_op["진동RMS"] > th]
    false_alarm = over[over["일자"] < 80]
    print(f"\n[{name}] 초과 {len(over)}건 | 최초 {over['일자'].min()}일 "
          f"| 정상기간 오탐 {len(false_alarm)}건 {false_alarm['일자'].tolist()}")